# Intermediate 02 — Modes of Convergence

Practice notebook for the **"Modes of Convergence"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
# Part 1 — Convergence in Probability

The weakest of the three modes in this section. A sequence $X_n$ **converges in probability** to $X$, written $X_n \to X$, if for every $\varepsilon > 0$,

$$\lim_{n\to\infty} P(|X_n - X| > \varepsilon) = 0.$$

Intuitively: for large $n$, $X_n$ is *very likely* to be close to $X$. It is a statement about the **probabilities of deviations shrinking**, not about individual sample paths.

**PDF example (Chebyshev).** Let $X_1, X_2, \dots$ be i.i.d. with $E[X_i]=\mu$ and $\operatorname{Var}(X_i)=\sigma^2<\infty$, and $\bar{X}_n = \frac{1}{n}\sum_{i=1}^n X_i$. Chebyshev's inequality gives

$$P(|\bar{X}_n - \mu| > \varepsilon) \leq \frac{\operatorname{Var}(\bar{X}_n)}{\varepsilon^2} = \frac{\sigma^2/n}{\varepsilon^2} = \frac{\sigma^2}{n\varepsilon^2} \to 0.$$

So $\bar{X}_n \to \mu$. We now **estimate** $P(|\bar{X}_n - \mu| > \varepsilon)$ by Monte Carlo over many independent sequences and watch it decay like $1/n$.


In [ ]:
# Convergence in probability: sample mean of i.i.d. Uniform(0,1) -> 0.5
rng = np.random.default_rng(42)
mu, sigma2 = 0.5, 1/12          # Uniform(0,1) mean and variance
eps = 0.05                      # epsilon threshold
n_seqs = 20_000                 # independent sequences for MC estimate of the probability

# Grid of sample sizes n
ns = np.array([10, 50, 100, 500, 1000, 5000])
probs = []
for n in ns:
    # (n_seqs, n) matrix of uniforms; take the sample mean of each row
    means = rng.uniform(0, 1, size=(n_seqs, n)).mean(axis=1)
    p_hat = np.mean(np.abs(means - mu) > eps)
    probs.append(p_hat)

probs = np.array(probs)
chebyshev = sigma2 / (ns * eps**2)   # upper bound sigma^2/(n*eps^2)

print(f"{'n':>6} | {'MC P(|Xbar-mu|>eps)':>20} | {'Chebyshev bound':>15}")
print("-" * 50)
for n, p, c in zip(ns, probs, chebyshev):
    print(f"{n:6d} | {p:20.4f} | {c:15.4f}")

fig, ax = plt.subplots()
ax.plot(ns, probs, "o-", label=r"MC estimate  $P(|\bar{X}_n-\mu|>\varepsilon)$")
ax.plot(ns, chebyshev, "s--", color="tab:red", label=r"Chebyshev  $\sigma^2/(n\varepsilon^2)$")
ax.plot(ns, 1/ns, ":", color="tab:gray", label=r"$1/n$ reference")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("sample size  $n$"); ax.set_ylabel("probability")
ax.set_title(rf"Convergence in probability — Uniform(0,1) sample mean, $\varepsilon$={eps}")
ax.legend()
plt.show()


**Your turn:** Lower $\varepsilon$ to $0.01$. The deviation event becomes *stricter*, so you need larger $n$ before the probability collapses. Confirm the Chebyshev bound still holds (it is loose — the true decay is much faster than $1/n$).


---
# Part 2 — Almost Sure Convergence

The strongest of the three modes here. $X_n$ **converges almost surely** to $X$, written $X_n \to} X$, if

$$P\big(\{\omega : X_n(\omega) \to X(\omega)\ \text{as}\ n\to\infty\}\big) = 1.$$

For **almost every** outcome $\omega$, the *single sequence of numbers* $X_n(\omega)$ converges in the ordinary deterministic sense. This is pointwise convergence outside a null set.

**Remark (PDF).** Almost sure convergence implies convergence in probability, but **not** conversely. Below we plot many independent sample paths of $\bar{X}_n$ for i.i.d. Uniform(0,1) and watch **every** path settle onto $\mu = 0.5$ — the visual signature of a.s. convergence (the strong law of large numbers).


In [ ]:
# Almost sure convergence: each path is a single sequence Xbar_n that settles to mu
rng = np.random.default_rng(7)
mu = 0.5
n_max = 20_000
n_paths = 12

# Each row is ONE sequence; the running mean across that row is the path of Xbar_n
draws = rng.uniform(0, 1, size=(n_paths, n_max))
running = np.cumsum(draws, axis=1, dtype=float) / np.arange(1, n_max + 1)

fig, ax = plt.subplots()
for i in range(n_paths):
    ax.plot(np.arange(1, n_max + 1), running[i], lw=0.8, alpha=0.8)
ax.axhline(mu, color="black", ls="--", lw=1.5, label=r"$\mu = 0.5$")
ax.set_xscale("log")
ax.set_xlabel("n  (log scale)"); ax.set_ylabel(r"$\bar{X}_n(\omega)$")
ax.set_title(r"Almost sure convergence — 12 paths of $\bar{X}_n$ all settle onto 0.5")
ax.legend()
plt.show()

# Pick one path and quantify its terminal gap for increasing n
one = running[0]
for n in [10, 100, 1000, 10000, 20000]:
    print(f"n={n:6d}: |Xbar_n - mu| = {abs(one[n-1] - mu):.6f}")


**Your turn:** Replace the uniform parent with a heavy-tailed distribution, e.g. `rng.standard_cauchy(...)`. The sample mean of i.i.d. Cauchys does **not** converge (Cauchy has no mean). Do the paths still settle? What do you see instead?


---
# Part 3 — The Weak Law of Large Numbers is Convergence in Probability

The **weak law of large numbers (WLLN)** is exactly the statement $\bar{X}_n \to \mu$. Here we make the connection explicit: we repeat the $\varepsilon$-deviation experiment for several distributions and show the empirical probability $P(|\bar{X}_n - \mu| > \varepsilon)$ decays toward 0 in every case. The *rate* differs by parent, but the *limit* is always 0.

| Parent | $\mu$ | $\sigma^2$ |
|---|---|---|
| Bernoulli(0.3) | 0.3 | 0.21 |
| Uniform(0,1) | 0.5 | 1/12 |
| Exponential(1) | 1.0 | 1.0 |


In [ ]:
rng = np.random.default_rng(2024)
eps = 0.1
ns = np.array([20, 100, 500, 2000, 10000])
n_seqs = 30_000

parents = {
    "Bernoulli(0.3)":   lambda n: (rng.uniform(0,1,size=(n_seqs,n)) < 0.3).astype(float),
    "Uniform(0,1)":     lambda n: rng.uniform(0, 1, size=(n_seqs, n)),
    "Exponential(1)":    lambda n: rng.exponential(1.0, size=(n_seqs, n)),
}
mus = {"Bernoulli(0.3)": 0.3, "Uniform(0,1)": 0.5, "Exponential(1)": 1.0}

results = {}
for name, sampler in parents.items():
    p_seq = []
    for n in ns:
        means = sampler(n).mean(axis=1)
        p_seq.append(np.mean(np.abs(means - mus[name]) > eps))
    results[name] = np.array(p_seq)

print(f"eps = {eps}")
print(f"{'n':>6} | " + " | ".join(f"{name:>18}" for name in results))
print("-" * 70)
for j, n in enumerate(ns):
    row = f"{n:6d} | " + " | ".join(f"{results[name][j]:18.4f}" for name in results)
    print(row)

fig, ax = plt.subplots()
for name, p_seq in results.items():
    ax.plot(ns, p_seq, "o-", label=name)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("n"); ax.set_ylabel(r"$P(|\bar{X}_n - \mu| > \varepsilon)$")
ax.set_title(f"WLLN as convergence in probability (eps={eps})")
ax.legend()
plt.show()


**Your turn:** Halve $\varepsilon$ to $0.05$ and rerun. Roughly how much larger must $n$ be to reach the same deviation probability as before? (Chebyshev predicts a factor of $\sim 4\times$ — do you see it?)


---
# Part 4 — Convergence in Distribution

The weakest notion — and the one the **CLT** uses. $X_n$ **converges in distribution** to $X$, written $X_n \Rightarrow X$, if

$$\lim_{n\to\infty} F_{X_n}(x) = F_X(x)$$

at every $x$ where $F_X$ is continuous. This is purely about **distributions**, not about any shared probability space.

**PDF example (CLT preview).** For i.i.d. $X_i$ with mean $\mu$ and variance $\sigma^2$,

$$Z_n = \frac{\bar{X}_n - \mu}{\sigma/\sqrt{n}} \Rightarrow N(0,1).$$

We demonstrate this by **plotting the empirical CDF of $Z_n$ against the standard normal CDF** for increasing $n$. As $n$ grows the empirical CDF locks onto $\Phi$ — the Glivenko-Cantelli flavor of convergence in distribution.


In [ ]:
rng = np.random.default_rng(11)
# Parent: Exponential(1), heavily skewed, mean=1, variance=1
mu, sigma = 1.0, 1.0
n_sim = 50_000

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, n in zip(axes, [1, 10, 100]):
    # Z_n = (Xbar_n - mu) / (sigma/sqrt(n))
    means = rng.exponential(1.0, size=(n_sim, n)).mean(axis=1)
    z = (means - mu) / (sigma / np.sqrt(n))

    # Empirical CDF of Z_n
    z_sorted = np.sort(z)
    ecdf = np.arange(1, n_sim + 1) / n_sim
    ax.step(z_sorted, ecdf, where="post", lw=1.5, label=f"empirical CDF of $Z_n$, n={n}")

    # Standard normal CDF as the limit
    grid = np.linspace(-4, 4, 400)
    ax.plot(grid, stats.norm.cdf(grid), "--", color="tab:red", lw=2, label=r"$\Phi$ (N(0,1) CDF)")
    ax.set_xlim(-4, 4); ax.set_ylim(0, 1)
    ax.set_xlabel("z"); ax.set_title(f"n = {n}")
    ax.legend(loc="upper left")

fig.suptitle(r"Convergence in distribution — empirical CDF of $Z_n$ approaches $\Phi$", y=1.03)
plt.tight_layout(); plt.show()

# Kolmogorov-Smirnov distance vs Phi: should shrink with n
print(f"{'n':>6} | {'KS distance to N(0,1)':>22}")
print("-" * 34)
for n in [1, 5, 10, 50, 100, 500]:
    means = rng.exponential(1.0, size=(n_sim, n)).mean(axis=1)
    z = (means - mu) / (sigma / np.sqrt(n))
    D = stats.kstest(z, "norm").statistic
    print(f"{n:6d} | {D:22.4f}")


**Your turn:** Swap the parent for `rng.uniform(0, 1, ...)` (mean 0.5, variance 1/12). Re-run the KS table. Does the empirical CDF approach $\Phi$ faster or slower than the exponential parent? Why?


---
# Part 5 — Comparing the Three Modes

The three modes are nested in strength:

$$\text{a.s. convergence} \;\Longrightarrow\; \text{convergence in probability} \;\Longrightarrow\; \text{convergence in distribution}.$$

The reverse implications are **false**. A classic counterexample for "in probability $\not\Rightarrow$ a.s." is the **sliding window** indicator $X_n = \mathbf{1}\{\lfloor \log_2 n \rfloor \text{ changes}\}$: rare events whose probabilities go to 0, but which happen infinitely often almost surely. We simulate a discrete analog: independent Bernoulli events with $p_n \to 0$ but $\sum p_n = \infty$ (so by Borel-Cantelli they occur infinitely often a.s.).


In [ ]:
rng = np.random.default_rng(5)
# p_n = 1/n -> 0 (convergence in probability to 0)
# but sum 1/n diverges -> events happen infinitely often (NOT a.s. convergence)
N = 50_000
n_grid = np.arange(1, N + 1)
p_n = 1.0 / n_grid

# Simulate 5 independent sequences
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for s in range(5):
    hits = rng.uniform(size=N) < p_n
    axes[0].plot(n_grid, hits.astype(int) + 0.02*s, lw=0.5, alpha=0.7,
                 label=f"path {s+1}")
axes[0].set_xscale("log")
axes[0].set_ylabel("X_n  (0/1)")
axes[0].set_title(r"$X_n \sim \mathrm{Bernoulli}(1/n)$: $P(|X_n-0|>1/2) = 1/n \to 0$ (in probability), "
                  "but spikes keep appearing")
axes[0].legend(loc="upper right", fontsize=8)

# Running count of hits per path — grows like log(N)
for s in range(5):
    hits = rng.uniform(size=N) < p_n
    axes[1].plot(n_grid, np.cumsum(hits), lw=1.0, alpha=0.8)
axes[1].plot(n_grid, np.log(n_grid), "k--", lw=2, label=r"$\log n$ (expected count)")
axes[1].set_xscale("log"); axes[1].set_yscale("log")
axes[1].set_xlabel("n"); axes[1].set_ylabel("cumulative hits")
axes[1].set_title("Cumulative count of rare events grows ~ log(n) — they never stop")
axes[1].legend()
plt.tight_layout(); plt.show()

# Sanity: P(|X_n - 0| > 1/2) = P(X_n = 1) = 1/n -> 0  => convergence in probability to 0.
# But the path does not converge to 0 (it keeps spiking) => NOT a.s. convergence.
print("At n=1000:  P(X_n=1) =", 1/1000)
print("At n=10000: P(X_n=1) =", 1/10000)
print("Yet each path keeps producing 1s forever (Borel-Cantelli).")


**Your turn:** Replace $p_n = 1/n$ with $p_n = 1/n^2$. Now $\sum p_n < \infty$, so Borel-Cantelli says hits happen only finitely often — the sequence **does** converge a.s. to 0. Confirm this visually: the cumulative-hit count should plateau. Which mode now holds?


---
# Summary

| Mode | Notation | About | Implies |
|---|---|---|---|
| Convergence in probability | $X_n \to X$ | deviation *probabilities* shrinking | convergence in distribution |
| Almost sure convergence | $X_n \to} X$ | individual paths converging | convergence in probability |
| Convergence in distribution | $X_n \Rightarrow X$ | CDFs converging | (weakest — implies nothing stronger) |

The **WLLN** is convergence *in probability*; the **SLLN** is convergence *almost surely*; the **CLT** is convergence *in distribution*. Same data, three different lenses.


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. State the definition of **convergence in probability**. Let $X_n \sim \text{Bernoulli}(1/n)$. Show $X_n \to 0$ but $X_n$ does **not** converge to 0 almost surely. Which Borel-Cantelli lemma is relevant?

2. Let $X_i$ be i.i.d. Uniform(0,1) with $\mu=0.5$ and $\sigma^2=1/12$. Using Chebyshev's inequality, give an explicit upper bound on $P(|\bar{X}_n - \mu| > 0.1)$. Then simulate 50,000 sample means for $n = 50, 500, 5000$ and compare the empirical probability to the bound.

3. Almost sure convergence implies convergence in probability, but not conversely. Construct a simulation with $p_n = 1/n$ (sliding rare events) and verify visually that the sequence is *not* almost surely convergent even though $P(|X_n| > 1/2) = 1/n \to 0$. Then switch to $p_n = 1/n^2$ and confirm the a.s. behavior changes.

4. State the **central limit theorem** as a convergence-in-distribution statement. Let $X_i \sim \text{Exponential}(1)$. Simulate $Z_n = (\bar{X}_n - 1)/(1/\sqrt{n})$ for $n = 1, 10, 100$ with 50,000 replicates and overlay the empirical CDF against $\Phi$. Report the Kolmogorov-Smirnov distance for each $n$.

5. Explain in your own words why convergence in distribution does **not** imply convergence in probability. Give a concrete counterexample using two independent sequences $X_n \sim N(0,1)$ and $X \sim N(0,1)$ (i.i.d. and independent of each other) — does $X_n \Rightarrow X$ hold? Does $X_n \to X$?


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** $X_n \to X$ iff for every $\varepsilon>0$, $P(|X_n-X|>\varepsilon)\to 0$. For $X_n\sim\text{Bernoulli}(1/n)$, $P(|X_n-0|>1/2)=P(X_n=1)=1/n\to 0$, so $X_n\to0$. But $\sum_n 1/n = \infty$ and the events $\{X_n=1\}$ are independent, so the **second Borel-Cantelli lemma** says $X_n=1$ occurs infinitely often a.s.; the path does not converge to 0, hence not a.s. convergence.

**2.** Chebyshev: $P(|\bar{X}_n-\mu|>\varepsilon)\leq \sigma^2/(n\varepsilon^2) = (1/12)/(n\cdot 0.01) = 1/(0.12\,n)$.

```python
import numpy as np
rng = np.random.default_rng(0)
mu, var, eps = 0.5, 1/12, 0.1
for n in [50, 500, 5000]:
    means = rng.uniform(0, 1, size=(50_000, n)).mean(axis=1)
    p_emp = np.mean(np.abs(means - mu) > eps)
    p_cheb = var / (n * eps**2)
    print(f"n={n:5d}: empirical={p_emp:.4f}, Chebyshev bound={p_cheb:.4f}")
```
The bound holds in every case and is very loose — the empirical probability is far below the bound.

**3.** See the Part 5 code. With $p_n=1/n$, $\sum p_n=\infty$ and independent events $\Rightarrow$ infinitely many hits a.s. (Borel-Cantelli II), so no a.s. convergence. With $p_n=1/n^2$, $\sum p_n<\infty$ so by Borel-Cantelli I only finitely many hits occur a.s. and $X_n\to}0$.

```python
import numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(1)
N = 100_000; n_grid = np.arange(1, N+1)
fig, ax = plt.subplots(1, 2, figsize=(12,4))
for p_seq, label in [(1/n_grid, "1/n"), (1/n_grid**2, "1/n^2")]:
    cum = np.cumsum(rng.uniform(size=N) < p_seq)
    ax[0].plot(n_grid, cum, label=label)
ax[0].set_xscale("log"); ax[0].set_yscale("log")
ax[0].set_title("cumulative hits"); ax[0].legend()
plt.show()
```
The $1/n$ curve keeps growing; the $1/n^2$ curve plateaus.

**4.** CLT: $Z_n = (\bar{X}_n-\mu)/(\sigma/\sqrt{n}) \Rightarrow N(0,1)$ in distribution.

```python
import numpy as np
from scipy import stats
rng = np.random.default_rng(2)
mu, sigma = 1.0, 1.0
for n in [1, 10, 100]:
    means = rng.exponential(1.0, size=(50_000, n)).mean(axis=1)
    z = (means - mu) / (sigma/np.sqrt(n))
    D = stats.kstest(z, "norm").statistic
    print(f"n={n:3d}: KS distance to N(0,1) = {D:.4f}")
```
KS distance shrinks rapidly with $n$.

**5.** If $X_n$ and $X$ are **independent** $N(0,1)$ variables, then $F_{X_n}=F_X=\Phi$ for every $n$, so trivially $F_{X_n}(x)\to F_X(x)$ — convergence in distribution holds. But $|X_n - X|\sim |N(0,2)|$ for every $n$, so $P(|X_n-X|>\varepsilon)$ is a positive constant independent of $n$ and does **not** go to 0. Hence $X_n$ does not converge to $X$ in probability. Convergence in distribution says nothing about the joint behavior of $X_n$ and $X$; convergence in probability does.

</details>
